In [1]:
import os
import json
import requests


OLLAMA_SERVER = os.getenv('OLLAMA_SERVER', 'http://localhost:11434')

### Intel MacBookPro 2015
TARGET_MODEL = "qwen2.5-coder:0.5b"


def get_ollama_metadata(model_name=None):
    base_url = f"{OLLAMA_SERVER}/api"
    
    try:
        # 1. GET /api/tags - Lists all locally available models
        # Source: https://docs.ollama.com/api/tags
        tags_response = requests.get(f"{base_url}/tags")
        tags_response.raise_for_status()
        all_models = tags_response.json().get('models', [])
        
        print(f"--- Locally Installed Models ({len(all_models)}) ---")
        for m in all_models:
            print(f"- {m['name']} (Size: {m['size'] / 1e9:.2f} GB)")

        # 2. POST /api/show - Get specific metadata for a model
        # Source: https://ollama.com
        if model_name:
            print(f"\n--- Specific Metadata for: {model_name} ---")
            show_payload = {"name": model_name}
            show_response = requests.post(f"{base_url}/show", json=show_payload)
            show_response.raise_for_status()
            
            # Print specific details like modelfile, parameters, and template
            metadata = show_response.json()
            # print(json.dumps(metadata, indent=2))
            metadata_family = metadata['details']["family"]
            context_length = metadata['model_info'][f"{metadata_family}.context_length"]
            embedding_length = metadata['model_info'][f"{metadata_family}.embedding_length"]
            print(f"context length - {context_length}")
            print(f"embedding length - {embedding_length}")
            return metadata
            
    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to Ollama. Is it running on port 11434?")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")

# Usage: Replace 'llama3' with your specific model name
metadata = get_ollama_metadata(TARGET_MODEL)


--- Locally Installed Models (2) ---
- mistral:latest (Size: 4.37 GB)
- qwen2.5-coder:0.5b (Size: 0.40 GB)

--- Specific Metadata for: qwen2.5-coder:0.5b ---
context length - 32768
embedding length - 896
